# CNN phân loại quần áo — Bản học viên
    Mỗi cá nhân làm cả hai project trong 14 ngày. Đọc `DE_BAI.md` cạnh notebook.
    Tuyến thực hành: train → baseline → DL → validation → khóa lựa chọn → test → demo.
    Kết quả tham khảo và runtime phải lấy từ lần chạy thực tế. Không cần DL thắng ML.
    Tra cứu: `../appendix/README.md`. Mọi câu hỏi bên dưới cần câu trả lời trong báo cáo.

## 1. Môi trường và đường dẫn
    **Làm:** chọn GPU và chạy ô này. **Quan sát:** device, phiên bản và output folder.
    **Giải thích:** vì sao notebook dùng CPU vẫn đúng nhưng thời gian thay đổi?
    **Tiếp:** tải dữ liệu; không nâng cấp toàn bộ thư viện của cloud. Xem `../CLOUD.md` nếu thiếu package.

In [ ]:
import os, sys
from pathlib import Path
# Chạy tại root repo hoặc thư mục con; tìm root có shared/.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'shared/data.py').exists()), None)
if ROOT is None:
    raise RuntimeError('Giải nén/clone toàn bộ repo, rồi chuyển thư mục tới nơi có shared/. Xem CLOUD.md.')
os.chdir(ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
import numpy as np
import torch
import matplotlib.pyplot as plt
from shared.data import load_data, describe, normalized_images
from shared.baselines import run_baselines
from shared.utils import load_config, read_json
from shared.engine import fit, make_loader, evaluate, reload_model, predict_image, predict_text
from shared.metrics import plot_confusion, plot_history, show_errors
from shared.protocol import register_protocol, lock_selection, final_test
from shared.kiem_tra import check_model, check_padding
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.set_num_threads(4)
DATA_ROOT = Path(os.environ.get('AI4B_DATA_ROOT', 'data'))
OUTPUT_ROOT = Path(os.environ.get('AI4B_OUTPUT_ROOT', 'runs/my_project'))
SMOKE = os.environ.get('AI4B_SMOKE') == '1'  # Chỉ kiểm kỹ thuật, không dùng để báo cáo benchmark.
if SMOKE: OUTPUT_ROOT = OUTPUT_ROOT / 'smoke'
print('PyTorch', torch.__version__, '| device:', DEVICE)
print('Data:', DATA_ROOT.resolve(), '| Output:', OUTPUT_ROOT.resolve(), '| smoke:', SMOKE)

In [ ]:
PROJECT = 'cnn'
config = load_config(PROJECT)
if SMOKE: config['epochs'] = 1
data = load_data(PROJECT, DATA_ROOT, subset=512 if SMOKE else None)
print(describe(data))
assert 'test' not in data

## 2. Khám phá train và kiểm split
    **Ví dụ:** chọn mẫu train của từng lớp; validation chỉ dùng đo và phân tích sau train.
    **Làm:** ghi số mẫu, phân bố lớp và shape/range hoặc chiều dài câu. Đọc manifest nguồn.
    **Quan sát:** train/validation có ID tách biệt; NLP neutral hiếm.
    **Giải thích:** trùng nội dung giữa split có khác trùng sample ID không? Vì sao không tự chia lại UIT-VSFC?
    **Tiếp:** giữ nguyên split và chạy hai baseline.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for label, ax in enumerate(axes.flat):
    i = np.flatnonzero(data['train']['y'] == label)[0]
    ax.imshow(data['train']['x'][i], cmap='gray'); ax.set_title(data['labels'][label]); ax.axis('off')
print('Raw:', data['train']['x'].shape, data['train']['x'].dtype,
      data['train']['x'].min(), data['train']['x'].max())
print('Mean/std chỉ từ train:', data['preprocessing'])

## 3. Baseline hoàn chỉnh
    **Làm:** chạy majority và logistic regression; preprocessing/TF-IDF chỉ fit train.
    **Quan sát:** accuracy đi cùng số đúng/tổng, macro-F1, recall/F1 từng lớp và confusion matrix.
    **Giải thích:** majority có accuracy đáng kể nhưng macro-F1 thấp vì sao?
    Chọn hai cặp nhầm phổ biến (CNN chú ý Shirt; NLP chú ý neutral).
    **Tiếp:** lập bảng shape/parameter trước khi viết mạng.

In [ ]:
baselines = run_baselines(data, OUTPUT_ROOT / PROJECT / 'baselines')
for name, r in baselines['models'].items():
    m = r['validation']
    print(name, 'accuracy:', m['accuracy'], f"({m['correct']}/{m['total']})", 'macro-F1:', m['macro_f1'])
    print(m['per_class'])
plot_confusion(baselines['models']['logreg']['validation'], data['labels']);

## 4. Tự xây model và bước train
    **Làm:** hoàn thiện TODO trong ô dưới; sau đó bật `RUN_DL=True`.
    Viết bảng shape, công thức parameter trong báo cáo trước khi chạy.
    `fit` chỉ điều phối và lưu checkpoint; kiến trúc, forward, train step do bạn viết.
    Cross-entropy nhận logits. Đặt softmax ở inference.
    **Quan sát:** kiểm shape và số parameter trước khi train.
    **Giải thích:** vì sao `eval()` khác `no_grad()`? Vì sao phải xóa gradient mỗi batch?
    **Tiếp:** chạy hai cấu hình với cùng seed 0 và cùng budget.

In [ ]:
"""Điền TODO trong notebook hoặc module này. Không có lời giải dự phòng."""
import torch
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence
from shared.data import normalize
from shared.engine import unpack_batch


def build_vocab(texts, min_freq=2):
    # TODO N1: chỉ đếm token trong texts train; PAD=0, UNK=1.
    # Giữ tần suất >= min_freq; sort (-frequency, token), rồi gán ID từ 2.
    raise NotImplementedError('TODO N1: xây vocabulary từ train, không đọc validation/test.')


def encode_text(text, vocab, max_length=40):
    # TODO N2: normalize, split, truncate, UNK, pad; trả (list[int], length thật).
    # Input rỗng phải báo ValueError, không trả length=0 cho packed sequence.
    raise NotImplementedError('TODO N2: chuyển một câu thành ID và chiều dài thật.')


class FashionCNN(nn.Module):
    def __init__(self, variant):
        super().__init__()
        # TODO C1: hai khối Conv → ReLU → MaxPool; head theo variant flatten/gap.
        raise NotImplementedError('TODO C1: khai báo các layer CNN theo bảng shape trong đề.')

    def forward(self, x):
        # TODO C2: trả logits (N,10), không softmax trước cross-entropy.
        raise NotImplementedError('TODO C2: viết forward CNN.')


class SentimentLSTM(nn.Module):
    def __init__(self, config):
        super().__init__()
        # TODO N3: Embedding với padding_idx=0, LSTM một chiều, Linear 3 lớp.
        raise NotImplementedError('TODO N3: khai báo các layer LSTM.')

    def forward(self, ids, lengths):
        # Khung packed sequence để không đọc hidden state tại PAD:
        # embedded = ...
        # packed = pack_padded_sequence(embedded, lengths.cpu(),
        #                              batch_first=True, enforce_sorted=False)
        # _, (h_n, _) = ...
        # return ...  # dùng h_n[-1]
        raise NotImplementedError('TODO N4: embedding → packed sequence → hidden cuối → logits.')


def build_model(config):
    return FashionCNN(config['variant']) if config['project'] == 'cnn' else SentimentLSTM(config)


def train_one_epoch(model, loader, optimizer, criterion, device, clip=None):
    # TODO T1: train mode; với từng batch dùng unpack_batch(batch, device).
    # zero_grad → logits → loss → backward → kiểm gradient → clip (nếu có) → step.
    # Trả loss trung bình theo số mẫu; không dùng validation để update weight.
    raise NotImplementedError('TODO T1: hoàn thiện một epoch train; xem Appendix A và D.')

In [ ]:
RUN_DL = False
SEEDS = [0]  # Bài nộp chỉ bắt buộc seed 0.

In [ ]:
if RUN_DL:
    vocab_check = build_vocab(data['train']['x'], config['min_freq']) if PROJECT == 'nlp' else None
    for variant in config['variants']:
        cfg = {**config, 'variant': variant, 'seed': 0,
               'vocab_size': len(vocab_check) if vocab_check else None}
        model = build_model(cfg)
        batch = next(iter(make_loader(data, 'train', cfg, vocab_check, encode_text)))
        expected = (20490 if variant == 'flatten' else 5130) if PROJECT == 'cnn' else 64*len(vocab_check)+33475
        check_model(model, list(batch[:-1]), len(data['labels']), expected)
else:
    print('Dừng phần DL: hoàn thiện TODO model/forward/train rồi đặt RUN_DL=True; không có lời giải tự chạy.')

## 5. Thí nghiệm có kiểm soát
    **Làm:** CNN đổi Flatten → GAP; NLP đổi CE thường → CE có trọng số `N/(3*N_c)`.
    Giữ split, seed, thứ tự batch, optimizer, learning rate, batch size và số epoch.
    Khởi tạo lại mỗi cấu hình; không tiếp tục từ cấu hình trước. Resume dùng `last.pt` cùng config.
    **Quan sát:** loss hữu hạn, weight thay đổi, history, `best.pt` và `last.pt` sau mỗi epoch.
    **Giải thích:** số parameter giảm có đảm bảo F1 tăng? Class weighting đổi recall neutral và accuracy thế nào?
    Không so trực tiếp giá trị hai loại loss để chọn NLP model.
    **Tiếp:** phân tích validation, chưa mở test.

In [ ]:
summaries = []
if RUN_DL:
    register_protocol(OUTPUT_ROOT, PROJECT, config, SEEDS)
    for variant in config['variants']:
        for seed in SEEDS:
            cfg = {**config, 'variant': variant, 'seed': seed}
            path = OUTPUT_ROOT / PROJECT / variant / f'seed_{seed}'
            summary = fit(data, cfg, path, build_model, train_one_epoch, build_vocab, encode_text, device=DEVICE)
            summaries.append(summary)
            print(variant, seed, summary['n_params'], summary['validation']['macro_f1'], summary['train_seconds'])
            plot_history(read_json(path / 'history.json'))
            plt.show()

## 6. Phân tích ít nhất 5 mẫu sai trên validation
    **Làm:** chọn từng cấu hình, mở confusion matrix và đọc mẫu sai bằng sample ID.
    **Quan sát:** CNN lỗi liên quan Shirt; NLP câu phủ định, câu vừa khen vừa chê, neutral và UNK.
    Nếu loại lỗi nào không có, ghi “không quan sát được” và chọn một lỗi thật khác.
    **Giải thích:** cho mỗi lỗi, viết giả thuyết và một cách kiểm giả thuyết; không tự đổi nhãn.
    **Tiếp:** chốt cấu hình theo macro-F1 validation, không chọn seed đẹp nhất.

In [ ]:
if RUN_DL:
    for variant in config['variants']:
        path = OUTPUT_ROOT / PROJECT / variant / 'seed_0'
        model, ck = reload_model(path / 'best.pt', build_model, 'cpu')
        loader = make_loader(data, 'validation', ck['config'], ck['vocab'], encode_text)
        result = evaluate(model, loader, 'cpu', data['labels'], data['validation']['ids'])
        print(variant, result['metrics']['per_class'])
        plot_confusion(result['metrics'], data['labels']); plt.show()
        errors = show_errors(data, result['ids'], result['pred'], n=5)
        if PROJECT == 'nlp': print(errors)
        else: plt.show()
        if PROJECT == 'nlp': check_padding(model, encode_text, ck['vocab'])

## 7. Khóa lựa chọn và test cuối (ngày 13)
    Quy tắc đã đăng ký trước train: mean macro-F1 validation, hòa ít parameter rồi thứ tự variant.
    `lua_chon.json` lưu checkpoint + SHA-256 trước khi có test metric.
    **Làm:** sau khi viết xong phân tích validation, bật `LOCK_SELECTION=True` và chạy.
    Sau đó mới bật `OPEN_TEST=True`. Chạy lại trả báo cáo của cùng khóa, không tune bằng test.
    Smoke test không được khóa hoặc báo cáo benchmark. Cờ là rào hướng dẫn quy trình, không phải phân quyền dữ liệu.
    **Quan sát:** `test_report.json` có accuracy đúng/tổng, macro-F1, metric từng lớp và confusion matrix.
    **Giải thích:** test cao hơn/thấp hơn validation cho phép kết luận tới mức nào?
    **Tiếp:** load lại checkpoint bằng CPU và demo.

In [ ]:
LOCK_SELECTION = False
OPEN_TEST = False
if RUN_DL and not SMOKE and LOCK_SELECTION:
    choice = lock_selection(OUTPUT_ROOT, PROJECT, SEEDS)
    print('Đã khóa:', choice['chosen']['variant'], choice['chosen']['mean_macro_f1'])
    if OPEN_TEST:
        report = final_test(OUTPUT_ROOT, PROJECT, DATA_ROOT, build_model, encode_text, open_test=True)
        print(report)
    else:
        print('Test vẫn đóng. Chỉ mở sau khi hoàn thiện phân tích và khóa lựa chọn.')

## 8. Demo sau khi load checkpoint trong phiên mới
    **Làm:** chọn checkpoint theo khóa (nếu chưa khóa, đây chỉ là demo kỹ thuật của cấu hình đầu).
    **Quan sát:** nhãn và điểm mọi lớp. Điểm softmax chưa phải xác suất đã hiệu chuẩn.
    **Giải thích:** Fashion-MNIST không chứng minh nhận diện ảnh điện thoại; câu tự gõ không tính benchmark.
    **Tiếp:** hoàn thiện báo cáo 2 trang và checklist bài nộp trong đề.

In [ ]:
if RUN_DL:
    choice_path = OUTPUT_ROOT / PROJECT / 'lua_chon.json'
    relative = read_json(choice_path)['chosen']['runs'][0]['checkpoint'] if choice_path.exists() else config['variants'][0]+'/seed_0/best.pt'
    model, ck = reload_model(OUTPUT_ROOT / PROJECT / relative, build_model, 'cpu')
    if PROJECT == 'cnn':
        image = data['validation']['x'][0]  # Ngoài train, đúng định dạng uint8 28x28.
        plt.imshow(image, cmap='gray'); plt.show()
        print(predict_image(model, image, ck['preprocessing'], ck['labels']))
    else:
        my_text = 'giảng viên nhiệt tình nhưng bài tập còn ít'
        print(predict_text(model, my_text, ck['vocab'], encode_text, ck['labels'], ck['config']['max_length']))

### Tự kiểm trước khi nộp
    Restart kernel → Run All; đủ baseline và hai cấu hình; có ít nhất 5 lỗi validation; đủ checkpoint,
    vocabulary (NLP), preprocessing và label mapping trong checkpoint; `lua_chon.json` trước test;
    báo cáo có số đúng/tổng, seed, môi trường, giới hạn. Demo cả hai bài tối đa 5 phút.